# Week 6 - Trees

In [1]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np

import seaborn as sns

from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt
%matplotlib inline

In [2]:
# GET DATA
#    import pandas as pd

data_url = 'https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-06-trees/CreditScoring.csv'

df = pd.read_csv(data_url)

In [3]:
# PREVIEW DATA
# df.head()

In [4]:
# PREPARE DATA
#   STANDARDIZE COLUMN NAMES
df.columns = df.columns.str.lower()


In [5]:
# REVIEW COLUMN NAMES AND DATATYPES
# df.dtypes

In [6]:
# CREATE A LIST OF CATEGORICAL COLUMN NAMES
# categorical_columns = df.select_dtypes(include=["object", "string"]).columns.tolist()  
# categorical_columns

In [7]:
# CREATE A LIST OF CATEGORICAL COLUMN NAMES
numerical_columns = df.select_dtypes(include="number").columns.tolist()
# numerical_columns

In [8]:
# SHOW FREQUENCE OF ALL UNIQUE VALUES PRESENT ON CATEGORICAL COLUMNS
# for c in categorical_columns:
#    print(df[c].value_counts())

In [9]:
# DESCRIBE STATISTICAL ATTRIBUTES OF EACH NUMERICAL COLUMN
# for n in numerical_columns:
#    print(df[n].describe)
    

In [10]:
# df.status.value_counts()

So 3200 customers paid back their loans, 1254 defaulted, and there's one record with the value 0 - an unknown status. We use dictionaries and the Pandas map method to translate the numbers into strings:

In [11]:
# MANUALLY CREATE A FINAL LIST OF CATEGOTICAL FEATURES
categorical_columns = ["status","home","marital","records","job"]

# MAP status_values CODES TO TEXT
status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

df.status = df.status.map(status_values)

# MAP home_values CODES TO TEXT
home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

df.home = df.home.map(home_values)

# MAP marital_values CODES TO TEXT
marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

df.marital = df.marital.map(marital_values)

# MAP records_values CODES TO TEXT
records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

df.records = df.records.map(records_values)

# MAP job_values CODES TO TEXT
job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df.job = df.job.map(job_values)

In [12]:
# PREVIEW DATAFRAME TO CONFIRM DATA PREPARATION IS SUFFICIENT TO MAKE IT USABLE
# df.head()

In [13]:
# CHECK FOR MISSING VALUES
#   MISSING VALUES PRESENT AS '99999999.0' IN THIS DATASET
# df.describe().round()

In [14]:
# CONVERT MISSING VALUES TO 'NaN'
for c in ['income', 'assets', 'debt']:
    df[c] = df[c].replace(to_replace=99999999, value=np.nan)

# PREVIEW MODIFIED DATAFRAME
df.describe().round()

#   NOTE: All 'count' ROW VALUES SHOULD BE THE SAME, COLUMNS WITH MISSING VALUES WILL DIFFER
#         ALL '99999999.0' HAVE NOW BEEN REMOVED, YOU CAN SEE THIS IN THE 'max' ROW

,seniority,time,age,expenses,income,assets,debt,amount,price
count,4455.0,4455.0,4455.0,4455.0,4421.0,4408.0,4437.0,4455.0,4455.0
mean,8.0,46.0,37.0,56.0,131.0,5403.0,343.0,1039.0,1463.0
std,8.0,15.0,11.0,20.0,86.0,11573.0,1246.0,475.0,628.0
min,0.0,6.0,18.0,35.0,0.0,0.0,0.0,100.0,105.0
25%,2.0,36.0,28.0,35.0,80.0,0.0,0.0,700.0,1118.0
50%,5.0,48.0,36.0,51.0,120.0,3000.0,0.0,1000.0,1400.0
75%,12.0,60.0,45.0,72.0,165.0,6000.0,0.0,1300.0,1692.0
max,48.0,72.0,68.0,180.0,959.0,300000.0,30000.0,5000.0,11140.0


In [15]:
# THERE IS ONLY ONE ROW WITH AN 'unk' STATUS 
#  WE ONLY CARE ABOUT THE 'okay' AND 'default' CLASSES
#  WE SHOULD DROP THE 'unk' ROW FROM OUT DATASET

df = df[df.status != 'unk'].reset_index(drop=True)

In [16]:
# SPLIT DATASET
# from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df,            test_size=0.2,  random_state=11)
df_train, df_val       = train_test_split(df_full_train, test_size=0.25, random_state=11)

In [17]:
# RESET THE INDEX VALUES FOR EACH DATAFRAME
df_train = df_train.reset_index(drop=True)
df_val   = df_val.reset_index(  drop=True)
df_test  = df_test.reset_index( drop=True)

# Preparing the target variable
The models give us a probability, so the target needs to be a number. We encode status: 0 for ok, 1 for default:


In [18]:
# SET TARGET COLUMN
target_column = "status"

# CREATE TARGET VECTOR
y_train = (df_train[target_column] == 'default').astype('int').values
y_val   = (df_val[target_column]   == 'default').astype('int').values
y_test  = (df_test[target_column]  == 'default').astype('int').values

# DROP TARGET FROM THE FEATURE MATRIX
for df in (df_train, df_val, df_test):
    df.drop(columns=[target_column], inplace=True)

# Decision trees
In this unit we train our first decision tree with scikit-learn on the credit scoring data, see what overfitting looks like, and learn how to control the size of a tree. \
 \ 
## What a decision tree looks like
At its core, a decision tree is just a set of if/else rules. For our credit risk problem, we could write one by hand:

```
def assess_risk(client):
    if client['records'] == 'yes':
        if client['job'] == 'parttime':
            return 'default'
        else:
            return 'ok'
    else:
        if client['assets'] > 6000:
            return 'ok'
        else:
            return 'default'
```

Let's take one customer from the training data and check what the rules say about them:


In [19]:
# DEFINE assess_risk() FUNCTION
def assess_risk(client):
    if client['records'] == 'yes':
        if client['job'] == 'parttime':
            return 'default'
        else:
            return 'ok'
    else:
        if client['assets'] > 6000:
            return 'ok'
        else:
            return 'default'
            
# EVALUATE RISK            
xi = df_train.iloc[0].to_dict()
assess_risk(xi)


'ok'

Writing rules by hand doesn't scale. The point of the decision tree algorithm is that it learns these rules from data automatically.
# Training a decision tree

We use DecisionTreeClassifier from scikit-learn. \
The features are mixed: numeric columns and categorical ones we encoded as strings. \
We turn each row into a dictionary, \
fill the missing values with 0, and \
use DictVectorizer to convert the dictionaries into a feature matrix:

In [20]:
from sklearn.tree               import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics            import roc_auc_score
from sklearn.tree               import export_text

In [21]:
train_dicts = df_train.fillna(0).to_dict(orient='records')

dv          = DictVectorizer(sparse=False)
X_train     = dv.fit_transform(train_dicts)

**DictVectorizer**
* one-hot encodes the categorical features
  - creates columns like job=partime and records=no 
 
and leaves the numeric ones unchanged.

In [22]:
# TRAIN THE DECISION TREE
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of sampl

In [23]:
# REVIEW MODEL AGAINST THE VALIDATION DATASET
val_dicts = df_val.fillna(0).to_dict(orient='records')
X_val     = dv.transform(val_dicts)

y_pred    = dt.predict_proba(X_val)[:, 1]
roc_auc_score(y_val, y_pred)

0.6553274321280667

In [24]:
# print(X_val, y_val)

In [25]:
# !!! THERE IS A PROBLEM HERE - AUC SHOULD BE '0.6548400377806302' !!!
y_pred = dt.predict_proba(X_train)[:, 1]
roc_auc_score(y_train, y_pred)

1.0

AUC of 1.0 on training but 0.65 on validation: the tree memorized the training data. \
It learned one specific rule per customer, so on the customers it has seen it's perfect \
these rules don't generalize to unseen customers. \
\
This is overfitting: memorizing the data but failing to generalize. \
\
The reason is that we let the tree grow without limits, so it became very deep: \
it kept splitting until every leaf was pure. 
## Controlling the size of the tree
We can restrict the depth of the tree with the max_depth parameter. \
A shallow tree cannot memorize each customer - it has to learn broader rules:

In [26]:
dt = DecisionTreeClassifier(max_depth=2)
dt.fit(X_train, y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",2
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples 

In [27]:
y_pred = dt.predict_proba(X_train)[:, 1]
auc = roc_auc_score(y_train, y_pred)
print('train:', auc)

y_pred = dt.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, y_pred)
print('val:', auc)

train: 0.7054989859726213
val: 0.6685264343319367


## Looking inside the tree

To see the rules the tree learned, we use export_text:



In [28]:
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

|--- records=yes <= 0.50
|   |--- job=partime <= 0.50
|   |   |--- class: 0
|   |--- job=partime >  0.50
|   |   |--- class: 1
|--- records=yes >  0.50
|   |--- seniority <= 6.50
|   |   |--- class: 1
|   |--- seniority >  6.50
|   |   |--- class: 0



A Decision Tree with only one question is called a **'DECISION STUMP'** \
\
The question is: 
- how does the learning algorithm decide which condition and
- which threshold to use at each split? 

# Tree vocabulary

First, the names for the parts of a tree:

    A **node** contains a condition, for example assets > 3000.
    Two **branches** go out of it: one for "true" and one for "false".
    The node at the top is the **root**; every node except the leaves is a **parent** of two other nodes.
    The nodes at the bottom are **leaves (decision nodes)**: this is where the tree stops and the prediction is made.
    The **depth of a tree** is the number of levels - the length of the longest path from the root to a leaf.

To understand how the algorithm learns such a tree, we take a small toy dataset instead of the full credit scoring data.
Finding the best split for one column \
 \
We take eight customers with one feature, assets, and the target, status:

In [30]:
data = [
    [8000, 'default'],
    [2000, 'default'],
    [   0, 'default'],
    [5000, 'ok'],
    [5000, 'ok'],
    [4000, 'ok'],
    [9000, 'ok'],
    [3000, 'default'],
]

df_example = pd.DataFrame(data, columns=['assets', 'status'])

In [32]:
# BUILD A DECISION STUMP WITH ONE CONDITION (e.g., assets > T)

# DEFINE THRESHOLDS (T) TO EVALUATE WHEN CONSIDERING APPROPRIATE T VALUES
Ts = [0, 2000, 3000, 4000, 5000, 8000]

In [33]:
T        = 4000
df_left  = df_example[df_example.assets <= T]
df_right = df_example[df_example.assets > T]

display(df_left)
print(df_left.status.value_counts(normalize=True))
display(df_right)
print(df_right.status.value_counts(normalize=True))

,assets,status
1,2000,default
2,0,default
5,4000,ok
7,3000,default


status
default    0.75
ok         0.25
Name: proportion, dtype: float64


,assets,status
0,8000,default
3,5000,ok
4,5000,ok
6,9000,ok


status
ok         0.75
default    0.25
Name: proportion, dtype: float64


# ADD ANOTHER FEATURE

In [35]:
data = [
    [8000, 3000, 'default'],
    [2000, 1000, 'default'],
    [   0, 1000, 'default'],
    [5000, 1000, 'ok'],
    [5000, 1000, 'ok'],
    [4000, 1000, 'ok'],
    [9000,  500, 'ok'],
    [3000, 2000, 'default'],
]

df_example = pd.DataFrame(data, columns=['assets', 'debt', 'status'])

In [36]:
# DEFINE THRESHOLDS TO TEST
thresholds = {
    'assets': [0, 2000, 3000, 4000, 5000, 8000],
    'debt': [500, 1000, 2000]
}

for feature, Ts in thresholds.items():
    print('#####################')
    print(feature)
    for T in Ts:
        print(T)
        df_left = df_example[df_example[feature] <= T]
        df_right = df_example[df_example[feature] > T]

        display(df_left)
        print(df_left.status.value_counts(normalize=True))
        display(df_right)
        print(df_right.status.value_counts(normalize=True))

        print()
    print('#####################')

#####################
assets
0


,assets,debt,status
2,0,1000,default


status
default    1.0
Name: proportion, dtype: float64


,assets,debt,status
0,8000,3000,default
1,2000,1000,default
3,5000,1000,ok
4,5000,1000,ok
5,4000,1000,ok
6,9000,500,ok
7,3000,2000,default


status
ok         0.571429
default    0.428571
Name: proportion, dtype: float64

2000


...

status
default    1.0
Name: proportion, dtype: float64

2000


,assets,debt,status
1,2000,1000,default
2,0,1000,default
3,5000,1000,ok
4,5000,1000,ok
5,4000,1000,ok
6,9000,500,ok
7,3000,2000,default


status
ok         0.571429
default    0.428571
Name: proportion, dtype: float64


,assets,debt,status
0,8000,3000,default


status
default    1.0
Name: proportion, dtype: float64

#####################


For debt, the best split gives an average impurity of 16% - worse than the 10% we got with assets. \
\
So the best split overall is still assets > 3000. With more features we would simply add more rows to this comparison.\

```
# PSEUDO CODE TO FIND THE BEST SPLIT THRESHOLD
FOR each feature in FEATURES:
    FIND all thresholds for the feature
    FOR each threshold in thresholds:
        SPLIT the dataset using "feature > threshold" condition
        COMPUTE the impurity of this split
SELECT the condition with the LOWEST IMPURITY
```
The misclassification rate is not the only impurity measure. \
Scikit-learn uses more sensitive criteria: Gini impurity and entropy. \
For regression trees the equivalent is MSE. \
\
The idea stays the same: pick the split with the lowest impurity. \
\
And while we looked at classification here, decision trees can also solve regression problems.

## Stopping criteria

After the root is split, the algorithm applies the same procedure recursively to the left and right sides. \
\
When does it stop?

- The group is already pure - the misclassification rate is 0%, no point in splitting further.
- The tree reached the maximum depth limit - controlled by max_depth.
- The group is too small to split - controlled by min_samples_leaf.
- The maximum number of leaves (decision nodes) was reached.

These stopping criteria are what keep a tree from overfitting.

    If the stopping criteria are not met (max depth not reached, groups large enough and not pure), repeat for the left side and the right side.


## Notes

This lesson first reviews the topics learned in the previous lesson about how to train a decision tree using scikit-learn, and handle a decision tree model not generalizing well due to overfitting of the data.

In this lesson, we learn about how to best split a decision tree and different classification criteria that can be used to split a tree. We dive deep in an example, splitting trees with misclassification criteria. Additionally, different stopping criteria to break the iterative tree split are discussed.

- **Structure of a decision tree**: A decision tree is a data structure composed of nodes (which contain conditions) and branches (which represent the values for a condition: True or False). The tree starts with a root node, which is the parent of two other nodes, and each of these nodes can also be the parent of others. At the last level of the tree, there are terminal nodes, also called leaves.

- **Depth of a decision tree**: The depth of a tree is the number of levels it has, or simply the length of the longest path from the root node to a leaf node.

- **Rules & Conditions, Thresholds**: The learning algorithm for a decision tree involves determining the best conditions to split the data at each node in order to achieve the best possible classifier. When there are many features, the algorithm considers each feature with its optimized threshold to determine the best feature for splitting at a particular node. In essence, at each node, the algorithm evaluates all possible thresholds for every feature and calculates the resulting misclassification rate. It then selects the condition (feature and threshold) that yields the lowest impurity.

- **Misclassification rate**: After each split, the goal is to divide the data into two sets that are as pure as possible. This means that the data within each set should belong predominantly to either one class, or the other. Another way to describe this is to aim for the lowest possible misclassification rate (impurity). The misclassification rate is a weighted average of the error rates obtained after splitting the data into two sets. The predicted class for each set is determined by the majority class present in this set.

- **Impurity criteria**: Common misclassification rate measurements are GINI Impurity and Entropy. It is also possible to use MSE for regression problems.

- **Decision trees can be used to solve regression problems**: While we focused on decision tree classifiers, it's important to note that decision trees can also be applied to regression problems using decision tree regressors.

- **Stopping Criteria**: The process of recursively splitting the data at each child node eventually stops based on certain stopping criteria. These criteria prevent the model from overfitting and include:
        The group is already pure: 0% impurity.
        The maximum depth has been reached.
        The group is smaller than the minimum size set for groups.
        The maximum number of leaves/terminal nodes has been reached.

**Decision Tree Learning Algorithm in a Nutshell**

    At a node, find the best split.
    Stop if max_depth is reached.
    For each child node, if the node is sufficiently large and not pure, repeat the process from the beginning.
